# TMFold — membrane topology on 3D structures

Predicts transmembrane topology with **DeepTMHMM**, maps it onto a 3D structure from the **current AlphaFold Database (v6)**,
your own **AlphaFold 3 / AlphaFold Server** result, an experimental **PDB** entry, or an on-the-fly **ColabFold / ESMFold**
prediction, then **places the membrane**, checks that structure and topology agree, and exports everything.

| | MembraneFold | TMFold (this notebook) |
|---|---|---|
| Proteins per run | 1 | many (UniProt query or whole proteome) |
| Structure sources | AFDB, OmegaFold, PDB upload | AFDB v6 (exact-sequence match via UniParc), AlphaFold 3 / Server upload, PDB ID, ColabFold (AlphaFold2), ESMFold |
| Membrane placement | – | fitted membrane slab, oriented model (OPM-style DUM atoms) |
| Quality control | pLDDT colouring | TM-in-membrane and loop-side agreement scores, per-segment pLDDT, tilt, hydrophobicity, positive-inside rule, UniProt annotation overlap, PAE map |
| Outputs | zip of plots/PDB | summary CSV, HTML report, oriented PDBs, PyMOL & ChimeraX scripts, interactive HTML viewers |
| Resumable | – | yes (DeepTMHMM batches and structures are cached) |

**Run:** set the forms below, then *Runtime → Run all*. A GPU runtime (*Runtime → Change runtime type → T4 GPU*) is only needed for ESMFold or ColabFold predictions.

Please cite DeepTMHMM (Hallgren *et al.* 2022, doi:10.1101/2022.04.08.487609), AlphaFold DB (Varadi *et al.* 2024, *NAR* 52:D368) and whichever predictor you use.

## 1. Install

In [ ]:
!pip install -q "tmfold @ git+https://github.com/victorolatunde23-dev/DeepTMHMM_Pipeline"
import tmfold, torch
print("tmfold", tmfold.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. (Optional) BioLib token
DeepTMHMM runs on BioLib. Anonymous use works; for more generous limits add an API token from biolib.com as the Colab
secret `BIOLIB_TOKEN` (key icon in the left sidebar).

In [ ]:
import os
try:
    from google.colab import userdata
    if userdata.get("BIOLIB_TOKEN"):
        os.environ["BIOLIB_TOKEN"] = userdata.get("BIOLIB_TOKEN")
        print("BioLib token loaded.")
except Exception:
    print("Running DeepTMHMM anonymously.")

## 3. Sequences
`uniprot` downloads directly from UniProt: `accessions` (e.g. `P02945, P02724`), any search `query`
(e.g. `organism_id:9606 AND keyword:KW-0812 AND reviewed:true`), or a `proteome` ID (e.g. `UP000000625`).

In [ ]:
#@title Input { display-mode: "form" }
INPUT_MODE = "uniprot"  #@param ["uniprot", "upload", "paste"]
UNIPROT_SOURCE = "accessions"  #@param ["accessions", "query", "proteome"]
UNIPROT_INPUT = "P02945, P02724, P0A917"  #@param {type:"string"}
MAX_SEQUENCES = 50  #@param {type:"integer"}
REVIEWED_ONLY = True  #@param {type:"boolean"}
RUN_NAME = "tmfold_run"  #@param {type:"string"}
PASTED_FASTA = """
>my_protein
MLELLPTAVEGVSQAQITGRPEWIWLALGTALMGLGTLYFLVKGMGVSDPDAKKFYAITTLVPAIAFTMYLSMLLGYGLTMVPFGGEQNPIYWARYADWLFTTPLLLLDLALLVDADQGTILALVGADGIMIGTGLVGALTKVYSYRFVWWAISTAAMLYILYVLFFGFTSKAESMRPEVASTFKVLRNVTVVLWSAYPVVWLIGSEGAGIVPLNIETLLFMVLDVSAKVGFGLILLRSRAIFGEAEAPEPSAGDGAAATSD
"""

## 4. Structures
* `auto` — your uploaded structure if given, else the **AlphaFold DB** model (found by accession, or for pasted sequences by an
  exact-sequence lookup in UniParc), else a prediction with `PREDICTOR`.
* `PREDICTOR`: `esmfold` (Colab GPU, ~1 min/protein; falls back to the ESM Atlas API ≤400 aa without a GPU),
  `esmatlas`, `colabfold` (AlphaFold2 with MSAs; slower, run the install cell in section 5), or `none`.
* For **AlphaFold 3**: run your sequence on AlphaFold Server, download the result zip and upload it in section 5.

In [ ]:
#@title Structure options { display-mode: "form" }
STRUCTURE_SOURCE = "auto"  #@param ["auto", "afdb", "predict", "none"]
PREDICTOR = "esmfold"  #@param ["esmfold", "esmatlas", "colabfold", "none"]
INCLUDE_NON_TM = False  #@param {type:"boolean"}
COMPARE_UNIPROT = True  #@param {type:"boolean"}
#@markdown Experimental structures, as `protein_id=PDBID:chain`, comma separated (e.g. `P02945=1C3W:A`)
PDB_IDS = ""  #@param {type:"string"}
#@markdown Tick to upload AlphaFold 3 / AlphaFold Server zips or PDB/mmCIF files in the next section
UPLOAD_STRUCTURES = False  #@param {type:"boolean"}

## 5. Load sequences, optional uploads and ColabFold

In [ ]:
from pathlib import Path
from tmfold import fetch_uniprot, parse_fasta, validate

OUT = Path("/content") / RUN_NAME
if INPUT_MODE == "uniprot":
    proteins = fetch_uniprot(UNIPROT_SOURCE, UNIPROT_INPUT, MAX_SEQUENCES, REVIEWED_ONLY)
elif INPUT_MODE == "upload":
    from google.colab import files
    proteins = [p for f in files.upload().values() for p in parse_fasta(f.decode())]
else:
    proteins = parse_fasta(PASTED_FASTA)
proteins = validate(proteins)
print(f"{len(proteins)} sequences:", ", ".join(p.id for p in proteins[:20]), "…" if len(proteins) > 20 else "")

In [ ]:
user_structures = {}
for item in filter(None, [x.strip() for x in PDB_IDS.split(",")]):
    pid, _, ref = item.partition("=")
    code_, _, chain = ref.partition(":")
    user_structures[pid.strip()] = {"kind": "pdb", "value": code_.strip(), "chain": chain.strip() or None}

if UPLOAD_STRUCTURES:
    from google.colab import files
    updir = OUT / "uploads"
    updir.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():
        f = updir / name
        f.write_bytes(data)
        # match a file to a protein by its ID in the file name (AlphaFold Server lowercases job names)
        match = [p.id for p in proteins if p.id.lower() in name.lower()] or ([proteins[0].id] if len(proteins) == 1 else [])
        if not match:
            print(f"  ! could not match {name} to a protein ID — rename it to include the ID")
            continue
        kind = "af3" if name.endswith(".zip") else "file"
        user_structures[match[0]] = {"kind": kind, "value": str(f), "b_is_plddt": kind == "af3"}
        print(f"  {name} → {match[0]} ({kind})")
print("user structures:", user_structures or "none")

In [ ]:
#@title Install ColabFold (only runs when PREDICTOR = "colabfold")
if PREDICTOR == "colabfold":
    !pip install -q --no-warn-conflicts "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"
    !pip install -q --upgrade dm-haiku
    print("ColabFold installed")

## 6. Run

In [ ]:
from tmfold import Config, run

cfg = Config(structure_source=STRUCTURE_SOURCE, predictor=PREDICTOR, include_non_tm=INCLUDE_NON_TM,
             compare_uniprot=COMPARE_UNIPROT, user_structures=user_structures)
summary, results = run(proteins, OUT, cfg)
cols = ["id", "type", "n_TM_helices", "n_beta_strands", "structure_source", "mean_pLDDT_TM",
        "tm_in_slab", "loop_side_agreement", "agreement", "positive_inside_ok", "uniprot_TM_overlap", "flags"]
summary[[c for c in cols if c in summary]]

## 7. Explore one protein
**agreement** = `consistent` when ≥80 % of predicted membrane residues sit inside the fitted membrane *and* ≥80 % of loop
residues are on their predicted side. Segments are flagged for low pLDDT, lying outside the membrane, short helices or steep tilt.

In [ ]:
#@title Viewer { display-mode: "form", run: "auto" }
PROTEIN = ""  #@param {type:"string"}
COLOR_BY = "topology"  #@param ["topology", "plddt", "hydrophobicity"]
import pandas as pd
from IPython.display import display, Image
from tmfold.viz import viewer

pid = PROTEIN or next((k for k, r in results.items() if r.get("structure") is not None), next(iter(results)))
r = results[pid]
print(pid, "—", r["description"])
if r.get("structure") is not None:
    viewer(r["structure"], r["topology"], r["membrane"], color_by=COLOR_BY).show()
    print("red = TM helix · green = β-strand · dark blue = cytoplasmic · light blue = outside · orange = signal peptide;"
          " membrane planes: blue (cytoplasmic) / red (outer)")
    display(pd.DataFrame(r["segments"]))
for key in ("profile", "pae"):
    if key in r["files"]:
        display(Image(str(r["files"][key])))

## 8. Download

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive(str(OUT), "zip", OUT)
print("Open report.html inside the zip for the full report.")
files.download(zip_path)

In [ ]:
#@title (Optional) copy results to Google Drive
from google.colab import drive
drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/TMFold") / RUN_NAME
shutil.copytree(OUT, dest, dirs_exist_ok=True)
print("Copied to", dest)